# Create IHI Awards (Innovative Health Initiative, EU)

Creates IHI awards from the project factsheet pages on ihi.europa.eu.

**Prerequisites:** run `scripts/local/ihi_to_s3.py` to scrape + upload first.

**Data source:** ihi.europa.eu Drupal site, project-factsheets listing at `/projects-results/project-factsheets`. ~243 individual factsheet pages, each rendered server-side with fields: Total Cost, Start Date, End Date, Grant agreement number, Project coordinator + a Participants table with per-org EU funding.

**S3 location:** `s3a://openalex-ingest/awards/ihi/ihi_projects.parquet`

**IHI funder in OpenAlex:** funder_id 4320326631 · display_name "Innovative Medicines Initiative" (the OpenAlex entity covers both IHI and its predecessor IMI; IHI took over from IMI in 2022) · BE.

**Schema notes:**
- `funder_award_id` = the EU Grant Agreement Number when published (e.g. `831434`); fallback to `ihi-{slug}` for older IMI projects without a GA number in the factsheet.
- `title` = full project name from og:title meta tag.
- `description` = project abstract from og:description meta tag.
- `amount` = first "Total Cost" value (the IHI/IMI funding contribution; a second Total Cost on each factsheet is the consortium total including industry/EFPIA co-funding).
- `pi_given_name` / `pi_family_name` = the "Project coordinator" name, split via canonical split_name (degree suffixes stripped).
- `coordinator_org` = first row of the Participants table (Drupal renders coordinator org first).
- `start_date` / `end_date` from dd/mm/yyyy fields.
- end_year ~78% — some active projects have no end-date published yet.

provenance `ihi_factsheets`, priority 199.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ihi_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ihi/ihi_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.ihi_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.ihi_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.ihi_raw LIMIT 5;

## Step 1.6: Funder existence fail-fast

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi FROM openalex.common.funder WHERE funder_id = 4320326631;

## Step 2: Create IHI Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ihi_awards
USING delta
AS
WITH
ihi_funder AS (
    SELECT funder_id, display_name, ror_id, doi FROM openalex.common.funder WHERE funder_id = 4320326631
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'EUR' ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'research' as funding_type,    -- IHI/IMI projects are all multi-million EUR R&D consortia
        CAST(NULL AS STRING) as funder_scheme,
        'ihi_factsheets' as provenance,
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
        TRY_CAST(g.start_year AS INT) as start_year,
        TRY_CAST(g.end_year AS INT) as end_year,
        CASE
            WHEN g.pi_family_name IS NOT NULL OR g.coordinator_org IS NOT NULL THEN
                struct(
                    g.pi_given_name as given_name,
                    g.pi_family_name as family_name,
                    CAST(NULL AS STRING) as orcid,
                    struct(
                        g.coordinator_org as name,
                        CAST(NULL AS STRING) as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation,
                    CAST(NULL AS DATE) as role_start
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >>) as investigators,
        COALESCE(g.landing_url, 'https://www.ihi.europa.eu/projects-results/project-factsheets') as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.ihi_raw g
    CROSS JOIN ihi_funder f
    WHERE g.funder_award_id IS NOT NULL AND TRIM(CAST(g.funder_award_id AS STRING)) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ihi_factsheets' AND priority = 199;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    199 as priority
FROM openalex.awards.ihi_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_ihi_awards FROM openalex.awards.ihi_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, amount, currency, start_year, lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.ihi_awards LIMIT 10;

In [ ]:
%sql
-- §6.4a freq check
SELECT lead_investigator.given_name AS g, lead_investigator.family_name AS f, COUNT(*) AS n
FROM openalex.awards.ihi_awards GROUP BY 1,2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT
    COUNT(*) as total, COUNT(amount) as has_amount, COUNT(lead_investigator.family_name) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_inst, COUNT(start_date) as has_start, COUNT(end_date) as has_end,
    ROUND(try_divide(COUNT(amount) * 100.0, COUNT(*)), 1) as pct_amount,
    ROUND(try_divide(COUNT(end_date) * 100.0, COUNT(*)), 1) as pct_end
FROM openalex.awards.ihi_awards;

In [ ]:
%sql
SELECT start_year, COUNT(*) FROM openalex.awards.ihi_awards WHERE start_year IS NOT NULL GROUP BY 1 ORDER BY 1 DESC LIMIT 20;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as inst, COUNT(*) as n
FROM openalex.awards.ihi_awards WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1 ORDER BY n DESC LIMIT 20;